In [0]:
from pyspark.sql.functions import *

df_raw = spark.read \
.option("header", True) \
.option("inferSchema", True) \
.csv("/Volumes/managed_catalog/managed_schema/expense_volume/smart_home_energy_logs.csv")


df_raw.display()

device_id,device_name,room_id,room_name,timestamp,energy_kwh
101,AC,1,Living Room,2026-06-01T08:00:00.000Z,5.2
102,TV,1,Living Room,2026-06-01T09:00:00.000Z,1.5
103,Fan,2,Bedroom,2026-06-01T10:00:00.000Z,2.1
104,Refrigerator,3,Kitchen,2026-06-01T11:00:00.000Z,4.3
105,Washing Machine,3,Kitchen,2026-06-01T12:00:00.000Z,3.5
101,AC,1,Living Room,2026-06-02T18:00:00.000Z,6.1
102,TV,1,Living Room,2026-06-02T20:00:00.000Z,1.8
103,Fan,2,Bedroom,2026-06-02T14:00:00.000Z,2.4
104,Refrigerator,3,Kitchen,2026-06-02T21:00:00.000Z,4.6
105,Washing Machine,3,Kitchen,2026-06-02T16:00:00.000Z,3.2


In [0]:
df_raw.printSchema()

root
 |-- device_id: integer (nullable = true)
 |-- device_name: string (nullable = true)
 |-- room_id: integer (nullable = true)
 |-- room_name: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)



In [0]:
df_raw.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "managed_catalog.managed_schema.energy_bronze"
)

In [0]:
%sql
SELECT * 
FROM managed_catalog.managed_schema.energy_bronze;

device_id,device_name,room_id,room_name,timestamp,energy_kwh
101,AC,1,Living Room,2026-06-01T08:00:00.000Z,5.2
102,TV,1,Living Room,2026-06-01T09:00:00.000Z,1.5
103,Fan,2,Bedroom,2026-06-01T10:00:00.000Z,2.1
104,Refrigerator,3,Kitchen,2026-06-01T11:00:00.000Z,4.3
105,Washing Machine,3,Kitchen,2026-06-01T12:00:00.000Z,3.5
101,AC,1,Living Room,2026-06-02T18:00:00.000Z,6.1
102,TV,1,Living Room,2026-06-02T20:00:00.000Z,1.8
103,Fan,2,Bedroom,2026-06-02T14:00:00.000Z,2.4
104,Refrigerator,3,Kitchen,2026-06-02T21:00:00.000Z,4.6
105,Washing Machine,3,Kitchen,2026-06-02T16:00:00.000Z,3.2


In [0]:
df_clean = df_raw \
.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)


df_clean = df_clean.dropna()


df_clean.display()

device_id,device_name,room_id,room_name,timestamp,energy_kwh
101,AC,1,Living Room,2026-06-01T08:00:00.000Z,5.2
102,TV,1,Living Room,2026-06-01T09:00:00.000Z,1.5
103,Fan,2,Bedroom,2026-06-01T10:00:00.000Z,2.1
104,Refrigerator,3,Kitchen,2026-06-01T11:00:00.000Z,4.3
105,Washing Machine,3,Kitchen,2026-06-01T12:00:00.000Z,3.5
101,AC,1,Living Room,2026-06-02T18:00:00.000Z,6.1
102,TV,1,Living Room,2026-06-02T20:00:00.000Z,1.8
103,Fan,2,Bedroom,2026-06-02T14:00:00.000Z,2.4
104,Refrigerator,3,Kitchen,2026-06-02T21:00:00.000Z,4.6
105,Washing Machine,3,Kitchen,2026-06-02T16:00:00.000Z,3.2


In [0]:
df_clean.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "managed_catalog.managed_schema.energy_silver"
)

In [0]:
daily_summary = df_clean.groupBy(
    "room_name",
    to_date("timestamp").alias("date")
)\
.agg(
    round(
        sum("energy_kwh"),
        2
    ).alias("daily_energy_kwh")
)


daily_summary.display()

room_name,date,daily_energy_kwh
Bedroom,2026-06-01,2.1
Living Room,2026-06-05,8.4
Kitchen,2026-06-05,8.2
Kitchen,2026-06-03,8.2
Living Room,2026-06-01,6.7
Bedroom,2026-06-02,2.4
Living Room,2026-06-02,7.9
Living Room,2026-06-03,8.5
Living Room,2026-06-04,6.1
Kitchen,2026-06-04,8.5


In [0]:
daily_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "managed_catalog.managed_schema.daily_energy_summary"
)

In [0]:
weekly_summary = df_clean.groupBy(
    "room_name"
)\
.agg(
    round(
        sum("energy_kwh"),
        2
    ).alias("weekly_energy_kwh")
)


weekly_summary.display()

room_name,weekly_energy_kwh
Bedroom,13.3
Kitchen,49.2
Living Room,46.0


In [0]:
weekly_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "managed_catalog.managed_schema.weekly_energy_summary"
)

In [0]:
%sql
SELECT
device_name,
SUM(energy_kwh) AS total_usage
FROM managed_catalog.managed_schema.energy_silver
GROUP BY device_name
ORDER BY total_usage DESC;

device_name,total_usage
AC,36.4
Refrigerator,28.599999999999998
Washing Machine,20.599999999999998
Fan,13.3
TV,9.6


In [0]:
%sql
SELECT
device_name,
SUM(energy_kwh) AS total_usage
FROM managed_catalog.managed_schema.energy_silver
GROUP BY device_name
HAVING SUM(energy_kwh) > 25
ORDER BY total_usage DESC;

device_name,total_usage
AC,36.4
Refrigerator,28.599999999999998


In [0]:
daily_summary.write \
.mode("overwrite") \
.option("header", True) \
.csv(
"/Volumes/managed_catalog/managed_schema/expense_volume/daily_energy_report"
)